# 📚 Módulo RAG Multimodal Completo — SafeDrive AI

**Objetivo:** Implementar la versión final del Asistente Post-Accidente de SafeDrive AI en un único flujo de trabajo. 
Incluye variables de configuración global, ingesta multimodal (texto + imágenes), vectorización local y exportación para visualización 3D.

## 🗺️ Flujo del Notebook
1. Instalación de dependencias.
2. **Configuración Global** (Variables de hiperparámetros y rutas).
3. Ingesta Multimodal (Gemini 2.5 Flash Vision + pypdfium2).
4. Chunking Adaptativo.
5. Indexación en ChromaDB (Ollama Embeddings).
6. Pipeline RAG de Emergencias.
7. Pruebas de estrés y validación.
8. **Exportación a TensorFlow Embedding Projector** (`.tsv`).

## 🔧 Paso 1: Instalación de dependencias

In [1]:
!pip install langchain langchain-chroma langchain-ollama chromadb pypdfium2 google-genai pillow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------- -------------------- 1.8/3.8 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 10.8 MB/s  0:00:00
   ---------------------------------------- 0.0/805.5 kB ? eta -:--:--
   ---------------------------------------- 805.5/805.5 kB 10.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.8 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 11.5 MB/s  0:00:00

   ---------------------------------------- 0/8 [pypdfium2]
   ---------------------------------------- 0/8 [pypdfium2]
   ---------------------------------------- 0/8 [pypdfium2]
   ---------------------------------------- 0/8 [pypdfium2]
   ---------- ----------------------------- 2/8 [pyasn1]
   ---------- ---------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ Paso 2: Configuración Global (Hiperparámetros y Rutas)
Centralizamos todas las variables del sistema para facilitar su modificación y cumplir con los estándares de diseño de software.

In [ ]:
import os

# --- RUTAS DE DIRECTORIOS ---
DATA_DIR = "./data/dgt/"
CHROMA_DIR = "./chroma_db/dgt_multimodal"
COLLECTION_NAME = "corpus_dgt_seguridad"

# --- CONFIGURACIÓN DE MODELOS ---
OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen2.5:7b"
VISION_MODEL = "gemini-2.5-flash"

# --- CONFIGURACIÓN RAG (Chunking) ---
CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

print("✅ Configuración global cargada en memoria.")

## 📂 Paso 3: Ingesta Multimodal (Extracción de Texto + Análisis Visual)

In [ ]:


try:
    ai_client = genai.Client()
    HAS_GEMINI = True
    print("✅ Cliente de Visión (Gemini) activado.")
except Exception as e:
    HAS_GEMINI = False
    print("⚠️ API Key no detectada. Operando en modo Text-Only.")

def procesar_pdf_multimodal(ruta_pdf: str) -> list:
    print(f"📄 Procesando PDF: {Path(ruta_pdf).name}")
    documentos_generados = []
    pdf = pdfium.PdfDocument(ruta_pdf)
    
    for page_idx in range(len(pdf)):
        pagina = pdf[page_idx]
        texto_nativo = pagina.get_textpage().get_text_bounded() or ""
        descripcion_grafica = "OMITIR"
        
        if HAS_GEMINI:
            bitmap = pagina.render(scale=2)
            pil_img = bitmap.to_pil()
            temp_img_path = f"temp_page_{page_idx}.png"
            pil_img.save(temp_img_path)
            
            prompt_vision = (
                "Analiza esta página del boletín de tráfico o primeros auxilios. "
                "Si encuentras señales, croquis, diagramas de reanimación o tablas, "
                "descríbelos detalladamente de forma técnica en español. Si solo hay texto, responde: 'OMITIR'."
            )
            
            try:
                img_open = Image.open(temp_img_path)
                response = ai_client.models.generate_content(model=VISION_MODEL, contents=[img_open, prompt_vision])
                descripcion_grafica = response.text
            except Exception as e:
                descripcion_grafica = "OMITIR"
                
            if os.path.exists(temp_img_path): os.remove(temp_img_path)
        
        contenido_final = f"--- TEXTO DE LA PÁGINA ---\n{texto_nativo}\n"
        if "OMITIR" not in descripcion_grafica and descripcion_grafica.strip() != "":
            contenido_final += f"\n[ANÁLISIS MULTIMODAL DE GRÁFICOS Y SEÑALES]:\n{descripcion_grafica}"
            print(f"   📸 Gráfico detectado en pág {page_idx + 1}")

        doc = Document(
            page_content=contenido_final,
            metadata={"source": Path(ruta_pdf).name, "pagina": page_idx + 1}
        )
        documentos_generados.append(doc)
        
    return documentos_generados

os.makedirs(DATA_DIR, exist_ok=True)
docs_totales = []
archivos_pdf = [f for f in os.listdir(DATA_DIR) if f.endswith(".pdf")]

for archivo in archivos_pdf:
    ruta_completa = os.path.join(DATA_DIR, archivo)
    docs_totales.extend(procesar_pdf_multimodal(ruta_completa))
    
print(f"Total de páginas listas: {len(docs_totales)}")

## ✂️ Paso 4: Chunking Adaptativo

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

splits = text_splitter.split_documents(docs_totales)
print(f"✂️ Generados {len(splits)} fragmentos.")

## 🗄️ Paso 5: Indexación en ChromaDB Local

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"✅ ChromaDB guardado en: {CHROMA_DIR}")

## 🧠 Paso 6: Pipeline RAG y Prompt de Emergencias

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = OllamaLLM(model=LLM_MODEL, base_url=OLLAMA_BASE_URL)

system_prompt = """Eres SafeDrive AI, un asistente experto en seguridad vial, primeros auxilios y normativa de la DGT (Dirección General de Tráfico de España).
Tu objetivo es asistir a conductores que acaban de sufrir un accidente de tráfico.
Responde de forma clara, directa y empática, priorizando siempre la seguridad física y la protección legal del usuario.

Usa ÚNICAMENTE los siguientes fragmentos de contexto recuperado de los manuales oficiales de la DGT para responder a la pregunta.
Si la respuesta no está en el contexto o tienes dudas, di estrictamente: "Por seguridad, no puedo ofrecer asesoramiento sobre ese tema. Por favor, contacta con el 112 si es una emergencia médica."
NO inventes leyes, NO alucines procedimientos médicos y NO des consejos que contradigan el contexto proporcionado.

Contexto recuperado:
{context}

Pregunta del conductor: {question}

Respuesta de SafeDrive AI:"""

prompt = ChatPromptTemplate.from_template(system_prompt)

def format_docs(docs):
    return "\n\n".join([f"[Origen: {d.metadata['source']} | Pág: {d.metadata['pagina']}]\n{d.page_content}" for d in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("🚀 Pipeline listo.")

## 📋 Paso 7: Pruebas de Estrés (Casos TFG)

In [ ]:
def consultar_asistente(pregunta: str):
    print(f"\n🙋‍♂️ Conductor: {pregunta}")
    print("-" * 50)
    docs_recuperados = retriever.invoke(pregunta)
    print(f"🔍 Chunks recuperados: {len(docs_recuperados)}")
    for d in docs_recuperados:
        print(f"   - Fuente: {d.metadata['source']} (Pág. {d.metadata['pagina']})")
        
    respuesta = rag_chain.invoke(pregunta)
    print(f"\n🤖 SafeDrive AI:\n{respuesta}\n")
    print("=" * 80)

consultar_asistente("El otro conductor quiere que firme el parte en blanco para rellenarlo luego, ¿debo hacerlo?")
consultar_asistente("Tengo una hemorragia en el brazo tras el choque, ¿qué hago?")


## 📊 Paso 8: Exportar a TensorFlow Embedding Projector
Extrae los vectores y sus metadatos a formato `.tsv`. Sube estos dos archivos generados a https://projector.tensorflow.org/ para visualizar la separación semántica de la ley en 3D.

In [ ]:
!pip install scikit-learn matplotlib

In [ ]:
import csv

print("📦 Exportando vectores para TF Embedding Projector...")

# Extraemos todo el contenido de la colección
datos_coleccion = vectorstore.get(include=["embeddings", "metadatas", "documents"])

# Exportar embeddings.tsv (coordenadas matemáticas)
with open("embeddings.tsv", "w", newline="", encoding="utf-8") as f_vec:
    writer = csv.writer(f_vec, delimiter="\t")
    writer.writerows(datos_coleccion["embeddings"])

# Exportar metadata.tsv (textos y orígenes)
with open("metadata.tsv", "w", newline="", encoding="utf-8") as f_meta:
    writer = csv.writer(f_meta, delimiter="\t")
    writer.writerow(["Fuente", "Pagina", "Texto_Muestra"]) 
    
    for meta, doc in zip(datos_coleccion["metadatas"], datos_coleccion["documents"]):
        texto_limpio = doc.replace("\n", " ").replace("\t", " ")[:150] + "..."
        writer.writerow([meta.get("source", "Desconocido"), meta.get("pagina", 0), texto_limpio])

print("✅ ¡Archivos 'embeddings.tsv' y 'metadata.tsv' generados con éxito!")
print("👉 Ve a https://projector.tensorflow.org/ y sube estos dos archivos.")

In [ ]:
# ── Preview visual con matplotlib (opcional) ──────────────────────
# Reducción PCA 2D local para previsualizar los clusters sin abrir el projector

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

print("🎨 Generando proyección 2D de los vectores semánticos...")

# 1. Convertimos los embeddings a un array de NumPy (datos_coleccion ya se extrajo en el paso 8)
vectores = np.array(datos_coleccion["embeddings"])

# Extraemos la fuente (el PDF) de cada fragmento para colorear los puntos
fuentes = [meta.get("source", "Desconocido") for meta in datos_coleccion["metadatas"]]
fuentes_unicas = list(set(fuentes))

# 2. Aplicamos PCA para reducir de cientos de dimensiones a solo 2 (X e Y)
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(vectores)

# 3. Configuramos el lienzo del gráfico
plt.figure(figsize=(12, 8))
colores = plt.cm.get_cmap('tab10', len(fuentes_unicas))

# 4. Dibujamos los puntos agrupados por su documento de origen
for i, fuente in enumerate(fuentes_unicas):
    # Buscamos los índices de los fragmentos que pertenecen a esta fuente
    idx = [j for j, f in enumerate(fuentes) if f == fuente]
    
    # Dibujamos el cluster (nube de puntos)
    plt.scatter(
        coords_2d[idx, 0], 
        coords_2d[idx, 1], 
        alpha=0.7, 
        label=fuente, 
        color=colores(i),
        edgecolors='w',
        linewidth=0.5,
        s=60 # Tamaño del punto
    )

# 5. Estilizado del gráfico
plt.title("Visualización 2D de la Base de Conocimientos DGT (PCA)", fontsize=14, pad=15)
plt.xlabel("Componente Principal 1", fontsize=11)
plt.ylabel("Componente Principal 2", fontsize=11)

# Ponemos la leyenda fuera del gráfico para que no tape los puntos
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()

# ¡Mostramos la magia!
plt.show()